# Insurance Policy RAG System — v2.2 (LangGraph)

**v2.2 = v2.1 + fixes for CUDA out-of-memory on some questions (Colab T4).**

| Cause | Fix |
|---|---|
| v2.1 filled the context *past* 1,500 tokens (the chunk that crossed the budget was kept) and the `[Section …]` labels were not counted, so prompts reached ~2,180 tokens vs ~1,600 in the original notebook — on a T4 already ~95% full with Mistral fp16 + reranker + embedder | **Hard 1,500-token cap that includes the labels** → prompts are back to the original's size (best-case evidence coverage stays 98–100%) |
| Freed GPU memory stays fragmented between stages | `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` + `torch.cuda.empty_cache()` after reranking and after each answer |
| One OOM failed the whole answer | On OOM: free memory, retry once with the top-ranked chunks (≤1,000 tokens), print a warning |
| No visibility into GPU headroom | Memory report after loading; warning if any LLM layers were pushed to CPU |
| Still OOM after all of the above | `CONFIG["llm_load_in_4bit"] = True` (off by default — it changes the model's numbers slightly) |

---

## v2.1 notes

**v2.1 = v2 + targeted fixes for the answer-quality regression versus the original notebook.**
Each fix below was traced by rebuilding the policy text from the original notebook's printed
chunks (all 84 reproduced exactly), running v2's own parsing/chunking code on it, and measuring
how much of the evidence the original answers actually used v2 could still deliver.

| # | Cause (measured) | Effect | v2.1 fix |
|---|---|---|---|
| 1 | Neighbor dedup drops adjacent subchunks | 9 of 10 original contexts used adjacent subchunks; adjacent v2 chunks share only ~29% (one sentence), so dedup threw away mostly new text. Best-case evidence coverage fell to 64% / 67% / 84% / 92% on 4 questions | `dedup_neighbors=False` |
| 2 | Same chunk *count* (5) with smaller chunks (avg 181 vs 288 tok) | Context shrank from ~1,500 to ~900 tokens; pre/post-hospitalization capped at 84% even without dedup | Fill context to the original's **1,500-token budget** (≈8 chunks) |
| 3 | Candidate pool cut to fused top-20 | Reranker chose from ~3,600 tokens of candidates vs ~5,800 originally; when legs disagree only ~10 dense candidates survive | Rerank the full union of dense top-32 + BM25 top-32 |
| 4 | Per-chunk score floor (−5) | Not in the original; your v1 log shows it dropping chunks (Plus Benefit, Excl06) | Floor off; gate on best score unchanged |
| 5 | Section titles dropped their first word | Labels read "Benefit On Renewal…" for **Plus** Benefit, "Travel Secure" for **Overseas** Travel Secure, "Expenses…" for **Pre/Post-Hospitalization** — in the embedded text, BM25 text and the LLM's context labels | Title starts at the heading's first word, ends at the heading sentence |
| 6 | Double BOS (`<s><s>[INST]`) | Rendered chat template already contains `<s>`; v2 tokenized it with special tokens again | `add_special_tokens=False` + runtime assertion |
| 7 | "Refer Annexure C" parsed as a heading | Blocked real Annexure B detection; 17-token junk section | `refer` added to reference words |
| 8 | Tail-merge repeated the overlap sentence | Duplicate sentence inside merged chunks | Merge only the new sentences |

**Generation wording:** the prompt instructions are restored to the original notebook's exact text
(v2 had added "ignore unrelated sections" and "cite … for each fact"). The section-labeled context
blocks and clean answer extraction (v2's formatting improvements) are kept. Whether the chat
template or the raw prompt gives better answers could not be measured offline — Section 10.2 runs
that A/B on identical contexts.

**New:** Section 10.1 checks every answer's context against the evidence the *original* notebook's
answers used (embedded fixture), so the regression is measured, not eyeballed.

Pipeline: **ingest → parse sections → sentence-pack chunks → index (dense + BM25) → hybrid retrieve (union) → rerank → budgeted context → gate → answer / no-answer**


## 0. Setup and configuration

In [ ]:
# ================================================================
# 0) Install dependencies
# ================================================================
!pip install -q tiktoken sentence-transformers transformers faiss-cpu accelerate rank_bm25 langgraph pandas

import os
# v2.2: reduce CUDA memory fragmentation (the fix PyTorch's own OOM message suggests).
# Must be set before torch first touches the GPU: if torch was already imported in this
# runtime, use Runtime -> Restart session, then run from the top.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import re, json, torch, faiss, traceback
import numpy as np
import pandas as pd
from pathlib import Path
from typing import TypedDict, List

IN_COLAB = False
try:
    from google.colab import files  # noqa
    IN_COLAB = True
except ImportError:
    pass

print(f"Colab: {IN_COLAB} | CUDA: {torch.cuda.is_available()}")


In [ ]:
# ================================================================
# 0.1) Configuration — every tunable in one place
# ================================================================
CONFIG = {
    # Chunking
    "chunk_size_tokens": 300,
    "table_chunk_tokens": 600,     # larger budget for table-like sections
    "min_tail_tokens": 50,         # tails below this are MERGED into previous chunk
    "title_max_words": 12,         # v2.1: was 8, which cut table titles at "(Year"

    # Models
    "embed_model": "BAAI/bge-small-en-v1.5",
    "bge_query_prefix": "Represent this sentence for searching relevant passages: ",
    "use_query_prefix": True,      # A/B toggle
    "use_section_prefix": True,    # prepend the section label to embedded + BM25 text (A/B toggle)
    "reranker_model": "BAAI/bge-reranker-base",
    "llm_model": "mistralai/Mistral-7B-Instruct-v0.3",

    # Retrieval (v2.1): candidate pool sized to the ORIGINAL's candidate text volume.
    # Original: 20 candidates x ~290 tokens ~= 5,800 tokens. v2 chunks average ~180 tokens -> 32.
    # Every candidate in the union of both legs is reranked (v2 truncated the fused list to 20).
    "k_dense": 32,
    "k_bm25": 32,
    "rrf_k": 60,                   # now only orders the union before reranking

    # Context (v2.1): restore the ORIGINAL context size instead of a fixed chunk count.
    "context_token_budget": 1500,  # HARD cap incl. section labels (v2.2) = original prompt size
    "max_context_chunks": 10,
    "dedup_neighbors": False,      # v2.1: off — adjacent v2 chunks share only one sentence

    # Gate
    "rerank_score_threshold": -5.0,  # refuse only if the BEST reranker score is below this
    "context_score_floor": None,     # v2.1: off (the original had no per-chunk floor)

    # Generation
    "max_new_tokens": 1000,
    "use_chat_template": True,     # A/B toggle: False = original notebook's raw prompt format

    # Memory (v2.2)
    "oom_retry_context_tokens": 1000,  # on CUDA OOM, retry once with the top chunks up to this size
    "llm_load_in_4bit": False,         # last resort if OOM persists: ~4 GB instead of ~14.5 GB

    # Persistence
    "index_path": "policy_index.faiss",
    "chunks_path": "chunks.json",
}

NO_ANSWER_TEXT = "Not specified in the provided policy text."

def is_no_answer(ans: str) -> bool:
    """The original prompt wraps the refusal in curly quotes; accept either form."""
    return ans.strip().strip('"“”').strip() == NO_ANSWER_TEXT


## 1. Ingest and preprocess

In [ ]:
# ================================================================
# 1) Upload (Colab) or set a path, normalize, strip page headers
# ================================================================
SOURCE_PATH = None  # set a path string here to skip the upload dialog

if SOURCE_PATH is None and IN_COLAB:
    print("Upload your policy .txt file")
    uploaded = files.upload()
    SOURCE_PATH = list(uploaded.keys())[0]

assert SOURCE_PATH, "Set SOURCE_PATH to your policy .txt file"

text = Path(SOURCE_PATH).read_text(encoding="utf8", errors="ignore")
text = re.sub(r"\r\n?", "\n", text)

# NEW: strip repeated page headers/footers from the PDF->txt conversion.
# These appear mid-sentence in the body and previously spawned a phantom
# 'my' section. Add any other repeated header lines you find to this list.
PAGE_HEADER_PATTERNS = [
    r"HDFC ERGO General Insurance Company Limited",
    r"my:\s*Optima Secure\s*-\s*Prospectus",
]
for pat in PAGE_HEADER_PATTERNS:
    text = re.sub(pat, " ", text)

text = re.sub(r"\n{3,}", "\n\n", text)
text = re.sub(r"[ \t]{2,}", " ", text)
text = text.strip()
print(f"File: {SOURCE_PATH} | length after cleaning: {len(text):,} characters")


## 2. Section parsing (sequence-aware)

The v1 regex only matched headings at line starts; in this document most headings occur
**mid-line inside flowing text**, so 14+ real sections were lumped under one label. This parser
finds candidates anywhere, then filters with three heuristics:

1. **Reference rejection** — "under Section 3.1", "as set out in Section 4" are cross-references,
   not headings (preceding word check).
2. **Title-case check** — real headings are followed by a Title-Case or numeric token.
3. **Monotone ladder** — genuine section numbers only ever advance (1 → 1.1 → 2 → 2.1 …);
   any candidate that goes backwards is a reference.

Annexures (letter IDs) are tracked on their own advancing ladder.

In [ ]:
# ================================================================
# 2) Detect section headings
# ================================================================
sec_pat = re.compile(r"(Section)\s+(\d+(?:\.\d+)*)\.?\s+(\S+)")
anx_pat = re.compile(r"(Annexure|Appendix)\s+([A-Z])\b\s*[-.]?\s*(\S*)")
REF_WORDS = ("under", "per", "in", "to", "against", "with",
             "below", "above", "of", "and", "see", "refer")   # v2.1: + "refer" ("Refer Annexure C")

def num_tuple(s: str):
    return tuple(int(x) for x in s.split("."))

candidates = sorted(
    [(m.start(), "sec", m) for m in sec_pat.finditer(text)] +
    [(m.start(), "anx", m) for m in anx_pat.finditer(text)]
)

headings, last_num, last_anx = [], (), ""
for pos, kind, m in candidates:
    nxt = m.group(3)
    if nxt and not (nxt[0].isupper() or nxt[0].isdigit()):
        continue  # followed by lowercase -> in-sentence reference
    before = text[max(0, pos - 25):pos].lower().split()
    if before and before[-1] in REF_WORDS:
        continue  # "under Section 3.1" etc.
    if kind == "sec":
        n = num_tuple(m.group(2))
        if last_num and n <= last_num:
            continue  # goes backwards -> reference
        last_num = n
        # v2.1: the title starts AT the heading's first word (group 3); v2 started after it,
        # dropping "Plus", "Overseas", "Pre-Hospitalization", "Procedure", ... from every label
        headings.append({"start": pos, "section_id": m.group(2), "title_start": m.start(3)})
    else:
        # v2.1: "Section 15.1. Annexure A - ..." -> the Annexure name is that section's title
        if headings and pos - headings[-1]["start"] < 20:
            continue
        if last_anx and m.group(2) <= last_anx:
            continue
        last_anx = m.group(2)
        headings.append({"start": pos, "section_id": f"Annexure-{m.group(2)}", "title_start": m.end(2)})

print(f"Detected {len(headings)} sections")


In [ ]:
# ================================================================
# 2.1) VERIFY the section ladder before proceeding
#      (everything downstream depends on this being right)
# ================================================================
ladder = [h["section_id"] for h in headings]
print("Section ladder:")
print(" -> ".join(ladder))

# Expect a clean monotone sequence like:
# 1 -> 1.1 -> 2 -> 2.1 -> 2.2 -> ... -> 15.1 -> Annexure-A -> Annexure-B
# If sections you know exist are MISSING, check whether a page-header pattern
# splits the heading text, or add the offending preceding word to REF_WORDS.


In [ ]:
# ================================================================
# 2.2) Build sections with clean short titles
# ================================================================
TITLE_WORDS = CONFIG["title_max_words"]

def make_title(s: str, max_words: int) -> str:
    """First words of the heading, stopping at the end of the heading sentence
    (e.g. '... Claim (Year 1).'), but not at abbreviations like '(Rs.)' or 'i.'."""
    out = []
    for w in s.split()[:max_words]:
        out.append(w)
        if w.endswith(".") and not re.fullmatch(r"\(?[A-Za-z]{1,3}\.\)?", w):
            break
    return " ".join(out).strip(" .:-")

sections = []
for i, h in enumerate(headings):
    start = h["start"]
    end = headings[i + 1]["start"] if i + 1 < len(headings) else len(text)
    body = text[start:end].strip()
    title = make_title(text[h["title_start"]:end], TITLE_WORDS)
    sections.append({
        "section_id": h["section_id"],
        "section_title": title,
        "text": body,
    })

# Preamble before the first heading (product intro) becomes its own section
# (any non-trivial preamble is kept — dropping it would lose text)
if headings and headings[0]["start"] > 50:
    sections.insert(0, {
        "section_id": "0",
        "section_title": "Introduction",
        "text": text[:headings[0]["start"]].strip(),
    })
if not headings:
    print("WARNING: no sections detected; using whole document as one section.")
    sections = [{"section_id": "0", "section_title": "Document", "text": text}]

print(f"{len(sections)} sections ready")


## 3. Sentence-aware chunking

Raw token windows cut mid-number ("₹2,00,00,000" → ",00,000. All Insured…"). This packer splits
each section into sentences and packs them greedily to the token budget with **one-sentence
overlap**. Undersized tails are **merged into the previous chunk** rather than dropped (dropping
lost text). Table-like sections (the plan-comparison illustrations) get a larger budget so
comparison rows stay together, at the cost of reranker/embedder truncation beyond 512 tokens —
BM25 still covers their full text.

Each chunk stores two texts: `text` (raw, shown to the LLM) and `embed_text` (prefixed with
`Section {id} – {title}:`), so mid-section chunks embed with their topic and section numbers are
BM25-searchable from every chunk.

In [ ]:
# ================================================================
# 3) Chunking
# ================================================================
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

def tok_len(s: str) -> int:
    return len(enc.encode(s))

def split_sentences(t: str):
    # Split after . or ; when followed by uppercase / digit / bullet / rupee / paren
    parts = re.split(r"(?<=[.;])\s+(?=[A-Z0-9\u00b7\u20b9(])", t)
    return [p.strip() for p in parts if p.strip()]

def looks_like_table(sec_text: str) -> bool:
    # Plan-comparison illustrations: dense with rupee amounts and 'for <Plan>'
    rupee_hits = len(re.findall(r"\u20b9|Rs\.?\s*\d", sec_text))
    for_hits = len(re.findall(r"\bfor (Suraksha|Secure|Super|Global)", sec_text))
    return rupee_hits >= 8 and for_hits >= 4

def pack_section(sec: dict, max_tok: int, min_tail: int):
    sents = split_sentences(sec["text"])
    packed, cur, cur_tok, fresh = [], [], 0, 0   # fresh = sentences not yet in packed[-1]
    for s in sents:
        st = tok_len(s)
        if fresh and cur_tok + st > max_tok:
            packed.append(" ".join(cur))
            cur = [cur[-1]]              # one-sentence overlap
            cur_tok = tok_len(cur[0])
            fresh = 0
        cur.append(s)
        cur_tok += st
        fresh += 1
    if fresh:
        new = cur[len(cur) - fresh:]
        if packed and tok_len(" ".join(new)) < min_tail:
            # v2.1: merge only the NEW sentences; the carried overlap sentence already
            # ends packed[-1] (v2 appended it a second time)
            packed[-1] = packed[-1] + " " + " ".join(new)
        else:
            packed.append(" ".join(cur))
    return packed

final_chunks = []
for sec in sections:
    budget = CONFIG["table_chunk_tokens"] if looks_like_table(sec["text"]) \
             else CONFIG["chunk_size_tokens"]
    pieces = pack_section(sec, budget, CONFIG["min_tail_tokens"])
    for j, piece in enumerate(pieces, start=1):
        label = f"Section {sec['section_id']} - {sec['section_title']}"
        final_chunks.append({
            "section_id": sec["section_id"],
            "section_title": sec["section_title"],
            "subchunk_id": j,
            "token_count": tok_len(piece),
            "text": piece,                        # raw text -> LLM context
            "embed_text": f"{label}: {piece}",    # labeled text -> embeddings + BM25
        })

print(f"Chunking complete -> {len(final_chunks)} chunks from {len(sections)} sections")


In [ ]:
# ================================================================
# 3.1) Inspect chunks (bounded preview)
# ================================================================
num_chunks = len(final_chunks)
avg_tokens = sum(c["token_count"] for c in final_chunks) / max(num_chunks, 1)
n_sections = len({c["section_id"] for c in final_chunks})
print(f"Total chunks: {num_chunks} | avg tokens: {avg_tokens:.1f} | distinct sections: {n_sections}")

for i, c in enumerate(final_chunks[:5]):
    print(f"\n=== Chunk {i+1} | Sec {c['section_id']} - {c['section_title']} "
          f"| sub {c['subchunk_id']} | {c['token_count']} tok ===")
    print(c["text"][:400], "...")

df_chunks = pd.DataFrame([{
    "Section_ID": c["section_id"],
    "Title": c["section_title"][:40],
    "Sub": c["subchunk_id"],
    "Tokens": c["token_count"],
    "Preview": c["text"][:100].replace("\n", " "),
} for c in final_chunks])
df_chunks.head(10)


## 4. Indexing: dense (FAISS) + lexical (BM25)

Both indexes are built over `embed_text` (section-labeled), so section numbers and titles are
searchable in every chunk. `IndexFlatIP` (exact) is deliberate at this corpus size. Both
artifacts are persisted, separating ingestion from serving.

In [ ]:
# ================================================================
# 4) Dense embeddings + FAISS
# ================================================================
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer(CONFIG["embed_model"])

embed_texts = [c["embed_text"] if CONFIG["use_section_prefix"] else c["text"]
               for c in final_chunks]
embeddings = embed_model.encode(
    embed_texts, batch_size=32, convert_to_numpy=True, show_progress_bar=True
).astype("float32")

faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("FAISS index built:", index.ntotal)


In [ ]:
# ================================================================
# 4.1) BM25 lexical index
# ================================================================
from rank_bm25 import BM25Okapi

def bm25_tokenize(s: str):
    return re.findall(r"[a-z0-9.]+", s.lower())  # keeps '4.7.1' and 'excl06' intact

bm25 = BM25Okapi([bm25_tokenize(t) for t in embed_texts])
print("BM25 index built over", len(embed_texts), "chunks")


In [ ]:
# ================================================================
# 4.2) Persist index + chunks
# ================================================================
faiss.write_index(index, CONFIG["index_path"])
Path(CONFIG["chunks_path"]).write_text(json.dumps(final_chunks, ensure_ascii=False))
print(f"Saved {CONFIG['index_path']} and {CONFIG['chunks_path']}")

# To reload without re-ingesting:
# index = faiss.read_index(CONFIG["index_path"])
# final_chunks = json.loads(Path(CONFIG["chunks_path"]).read_text())
# embed_texts = [c["embed_text"] if CONFIG["use_section_prefix"] else c["text"] for c in final_chunks]
# bm25 = BM25Okapi([bm25_tokenize(t) for t in embed_texts])


## 5. Hybrid retrieval (dense + BM25) — full union is reranked

v2.1: v2 truncated the RRF-fused list to 20, so when the two legs disagreed only ~10 dense
candidates reached the reranker (the original reranked 20). Now both legs return 32 candidates
(sized so the candidate *text volume* matches the original's 20 × ~290 tokens) and the whole
union is reranked. RRF only orders the union.

In [ ]:
# ================================================================
# 5) Hybrid retrieval
# ================================================================
def hybrid_retrieve(query: str,
                    k_dense=CONFIG["k_dense"],
                    k_bm25=CONFIG["k_bm25"]):
    # Dense leg — BGE query instruction prefix (passages are embedded without it)
    q_text = (CONFIG["bge_query_prefix"] + query) if CONFIG["use_query_prefix"] else query
    q_emb = embed_model.encode([q_text], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    _, I = index.search(q_emb, k_dense)
    dense_ranked = [int(i) for i in I[0] if i != -1]

    # Lexical leg
    scores = bm25.get_scores(bm25_tokenize(query))
    bm25_ranked = [int(i) for i in np.argsort(scores)[::-1][:k_bm25]]

    # Reciprocal rank fusion — orders the union; nothing is cut (v2.1)
    K = CONFIG["rrf_k"]
    rrf = {}
    for rank, idx in enumerate(dense_ranked):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (K + rank)
    for rank, idx in enumerate(bm25_ranked):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (K + rank)

    fused = sorted(rrf, key=rrf.get, reverse=True)
    return [final_chunks[i] for i in fused]


## 6. Reranker and context selection

v2.1: the context is filled **in rerank order up to the original's size (1,500 tokens)** instead of a
fixed 5 chunks. v2's smaller chunks (avg ~180 vs ~290 tokens) made "top-5" ~40% less text than the
original's top-5. Neighbor dedup is off: in the original, 9 of 10 answered questions used adjacent
subchunks together (procedures and lists run across chunk boundaries), and adjacent v2 chunks share
only the one carried sentence.

In [ ]:
# ================================================================
# 6) Reranker + context selection
# ================================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
tok_rerank = AutoTokenizer.from_pretrained(CONFIG["reranker_model"])
reranker = AutoModelForSequenceClassification.from_pretrained(CONFIG["reranker_model"])
reranker.to(device).eval()

def rerank_all(query: str, docs: List[str], batch_size: int = 16):
    """Score every doc against the query (batched: the union can hold up to ~64 candidates)."""
    out = []
    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]
        inputs = tok_rerank([query] * len(batch), batch,
                            padding=True, truncation=True, max_length=512,
                            return_tensors="pt").to(device)
        with torch.no_grad():
            out.append(reranker(**inputs).logits.squeeze(-1).float().cpu().numpy().reshape(-1))
    scores = np.concatenate(out) if out else np.array([])
    order = scores.argsort()[::-1]
    return order, scores

def free_gpu():
    """Return cached blocks to the allocator between pipeline stages (v2.2)."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def block_label(c: dict) -> str:
    return f"[Section {c['section_id']} - {c['section_title']}]"

def block_tokens(c: dict) -> int:
    """Tokens this chunk adds to the prompt: label + newline + text + separator."""
    if "block_tokens" not in c:
        c["block_tokens"] = tok_len(block_label(c)) + c["token_count"] + 2
    return c["block_tokens"]

def _adjacent(a: dict, b: dict) -> bool:
    return a["section_id"] == b["section_id"] and abs(a["subchunk_id"] - b["subchunk_id"]) == 1

def select_context(candidates: List[dict], order, scores):
    """Walk best-first and fill the context up to the original pipeline's size.
    v2.2: the budget is a HARD cap and counts the section labels; a chunk that would push the
    context past it is skipped (smaller lower-ranked chunks can still fill the remaining space)."""
    budget, cap = CONFIG["context_token_budget"], CONFIG["max_context_chunks"]
    floor = CONFIG["context_score_floor"]
    kept, kept_scores, used = [], [], 0
    for i in order:
        c, s = candidates[i], float(scores[i])
        if floor is not None and s < floor:
            continue
        if CONFIG["dedup_neighbors"] and any(_adjacent(c, k) for k in kept):
            continue
        if used + block_tokens(c) > budget:
            continue
        kept.append(c)
        kept_scores.append(s)
        used += block_tokens(c)
        if len(kept) >= cap:
            break
    return kept, kept_scores


## 7. LLM

v2.1: the rendered Mistral chat template already starts with `<s>`; v2 tokenized that text with
special tokens again, so every prompt began `<s><s>[INST]`. The text is now tokenized with
`add_special_tokens=False` (one BOS is added only if the template lacks it), and an assertion
fails loudly if a prompt ever starts with two BOS tokens. `use_chat_template=False` reproduces the
original notebook's raw prompt format for the A/B test in Section 10.2. Only newly generated tokens
are decoded in both modes (no prompt echo).

In [ ]:
# ================================================================
# 7) LLM
# ================================================================
from transformers import AutoModelForCausalLM

tokenizer_llm = AutoTokenizer.from_pretrained(CONFIG["llm_model"])
tokenizer_llm.pad_token = tokenizer_llm.eos_token
if CONFIG["llm_load_in_4bit"]:
    try:
        import bitsandbytes  # noqa: F401
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
    from transformers import BitsAndBytesConfig
    model_llm = AutoModelForCausalLM.from_pretrained(
        CONFIG["llm_model"], device_map="auto",
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16))   # T4 has no bfloat16 support
else:
    model_llm = AutoModelForCausalLM.from_pretrained(
        CONFIG["llm_model"], torch_dtype=torch.float16, device_map="auto")
model_llm.eval()

def gpu_report(tag: str = ""):
    if not torch.cuda.is_available():
        return
    free, total = torch.cuda.mem_get_info()
    print(f"[GPU {tag}] in use {(total - free) / 1e9:.1f} / {total / 1e9:.1f} GB "
          f"| free {free / 1e9:.2f} GB")

gpu_report("after loading all models")
_offloaded = [k for k, v in (getattr(model_llm, "hf_device_map", None) or {}).items()
              if v in ("cpu", "disk")]
if _offloaded:
    print(f"WARNING: {len(_offloaded)} LLM modules were placed on CPU/disk because the GPU is full. "
          "Generation will be slow and OOM-prone; set CONFIG['llm_load_in_4bit'] = True.")

# Diagnostic: does the rendered template already carry BOS? (If True, v2's tokenization doubled it.)
_probe = tokenizer_llm.apply_chat_template(
    [{"role": "user", "content": "x"}], tokenize=False, add_generation_prompt=True)
print("Chat template text starts with BOS:", _probe.startswith(tokenizer_llm.bos_token or "<s>"))

def build_llm_inputs(user_msg: str, use_chat_template: bool = None):
    use_tpl = CONFIG["use_chat_template"] if use_chat_template is None else use_chat_template
    bos = tokenizer_llm.bos_token_id
    if use_tpl:
        prompt_text = tokenizer_llm.apply_chat_template(
            [{"role": "user", "content": user_msg}], tokenize=False, add_generation_prompt=True)
        enc_in = tokenizer_llm(prompt_text, return_tensors="pt", add_special_tokens=False)
        if enc_in["input_ids"][0, 0].item() != bos:      # template without BOS -> add exactly one
            enc_in = tokenizer_llm(prompt_text, return_tensors="pt")
    else:
        enc_in = tokenizer_llm(user_msg, return_tensors="pt")   # original notebook's format
    ids = enc_in["input_ids"][0]
    assert not (len(ids) > 1 and ids[0].item() == bos and ids[1].item() == bos), \
        "Prompt starts with two BOS tokens"
    return enc_in.to(model_llm.device)

def generate_answer(user_msg: str, use_chat_template: bool = None) -> str:
    enc_in = build_llm_inputs(user_msg, use_chat_template)
    with torch.no_grad():
        out_ids = model_llm.generate(
            **enc_in,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=False,
            pad_token_id=tokenizer_llm.eos_token_id,
        )
    return tokenizer_llm.decode(
        out_ids[0][enc_in["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


## 8. LangGraph pipeline

Gate (unchanged from v2): refuse only when the **best** reranker score is below −5.0. Your earlier
run's top scores ranged from −1.1 to 5.7 on answerable questions, so all of them pass.

v2.1 changes here: context = `select_context()` (token budget, no dedup, no per-chunk floor), and
the prompt instructions are the original notebook's exact text. The context keeps v2's
`[Section X - Title]` labels, now with correct titles.

In [ ]:
# ================================================================
# 8) LangGraph pipeline
# ================================================================
from langgraph.graph import StateGraph, END

class RAGState(TypedDict):
    query: str
    retrieved: List[dict]
    reranked: List[dict]
    scores: List[float]
    answer: str

# The ORIGINAL notebook's instructions, verbatim (v2 had changed two of these lines).
PROMPT_TEMPLATE = """You are a professional insurance policy analyst.

Use ONLY the information provided in the CONTEXT to answer the QUESTION.
Do NOT add, assume, or infer details beyond the given text.
Provide a clear, factual, and complete answer in full sentences.
When relevant, cite the section name or number from the CONTEXT.
If the answer cannot be found, reply: “Not specified in the provided policy text.”

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""

def format_context(chunks: List[dict]) -> str:
    return "\n\n".join(f"{block_label(c)}\n{c['text']}" for c in chunks)

def build_prompt(query: str, chunks: List[dict]) -> str:
    return PROMPT_TEMPLATE.format(context=format_context(chunks), query=query.strip())

def retrieve_node(state: RAGState):
    try:
        query = state.get("query", "").strip()
        if not query:
            return {"retrieved": []}
        return {"retrieved": hybrid_retrieve(query)}
    except Exception as e:
        print(f"[retrieve] failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        return {"retrieved": []}

def rerank_node(state: RAGState):
    try:
        retrieved = state.get("retrieved", [])
        if not retrieved:
            return {"reranked": [], "scores": []}
        docs = [r["text"] for r in retrieved]
        order, scores = rerank_all(state["query"].strip(), docs)
        kept, kept_scores = select_context(retrieved, order, scores)
        free_gpu()   # v2.2: hand reranker memory back before generation
        return {"reranked": kept, "scores": kept_scores}
    except Exception as e:
        print(f"[rerank] failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        return {"reranked": [], "scores": []}

def gate(state: RAGState) -> str:
    scores = state.get("scores", [])
    if scores and max(scores) >= CONFIG["rerank_score_threshold"]:
        return "answer"
    return "no_answer"

def answer_node(state: RAGState):
    try:
        chunks, scores = state.get("reranked", []), state.get("scores", [])
        if not chunks:
            return {"answer": NO_ANSWER_TEXT}
        try:
            return {"answer": generate_answer(build_prompt(state["query"], chunks))}
        except torch.cuda.OutOfMemoryError:
            # v2.2: free memory and retry once with the best-ranked chunks only
            free_gpu()
            keep, used = [], 0
            for c in chunks:                      # already in rerank order
                if keep and used + block_tokens(c) > CONFIG["oom_retry_context_tokens"]:
                    break
                keep.append(c)
                used += block_tokens(c)
            print(f"[answer] CUDA out of memory — retrying with the top {len(keep)} of "
                  f"{len(chunks)} chunks ({used} tokens)")
            gpu_report("before retry")
            ans = generate_answer(build_prompt(state["query"], keep))
            return {"answer": ans, "reranked": keep, "scores": scores[:len(keep)]}
    except Exception as e:
        print(f"[answer] failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        return {"answer": NO_ANSWER_TEXT}
    finally:
        free_gpu()

def no_answer_node(state: RAGState):
    return {"answer": NO_ANSWER_TEXT}

workflow = StateGraph(RAGState)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("rerank", rerank_node)
workflow.add_node("answer", answer_node)
workflow.add_node("no_answer", no_answer_node)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "rerank")
workflow.add_conditional_edges("rerank", gate,
                               {"answer": "answer", "no_answer": "no_answer"})
workflow.add_edge("answer", END)
workflow.add_edge("no_answer", END)

app = workflow.compile()
print("Pipeline ready: retrieve -> rerank -> budgeted context -> gate -> answer | no_answer")


## 9. Ask questions

In [ ]:
# ================================================================
# 9) Query helper — shows sources + scores for auditing
# ================================================================
def ask(query: str, show_sources: bool = True):
    result = app.invoke({"query": query})
    print("Q:", query.strip())
    if show_sources and result.get("reranked"):
        srcs = [f"{c['section_id']} ({s:.2f})"
                for c, s in zip(result["reranked"], result.get("scores", []))]
        print("Sources:", ", ".join(srcs))
        print("Context tokens:", sum(c["token_count"] for c in result["reranked"]))
    print("\nA:", result["answer"])
    print("=" * 80)
    return result

_ = ask("What expenses are covered under the Overseas Travel Secure option? Give complete details.")


In [ ]:
# The full original query set
_ = ask("What is the difference between Optima Secure and Optima Super Secure plans? Give complete details.")
_ = ask("What are the standard exclusions under the my:Optima Secure policy? Give complete details.")
_ = ask("What are the exact specific conditions with a 24-month waiting period? Give complete details.")
_ = ask("Under what circumstances can obesity-related surgery be covered under Excl06, and what certifications are required?")
_ = ask("How to file reimbursement claims? Give me step wise complete procedure.")
_ = ask("What is the procedure for cashless claims in India? Give me step wise complete procedure.")
_ = ask("What is the Plus Benefit? Explain in detail.")
_ = ask("What are the exact inclusions for pre-hospitalization expenses? Give complete details.")
_ = ask("What are the exact inclusions for post-hospitalization expenses? Give complete details.")


## 10. Evaluation harness

Section IDs in `GOLD` were verified against the section ladder this parser produces on your policy
document. Run after **every** change to chunking, retrieval, thresholds, or prompts.

In [ ]:
# ================================================================
# 10) Gold-set evaluation
# ================================================================
GOLD = [
    {
        "query": "What expenses are covered under the Overseas Travel Secure option? Give complete details.",
        "gold_sections": {"4.11"},
        "must_contain": ["5 consecutive days", "accompanying person", "incurred and paid overseas"],
    },
    {
        "query": "What is the Plus Benefit? Explain in detail.",
        "gold_sections": {"4.4", "4.4.1"},
        "must_contain": ["100%", "2 years", "irrespective of a claim"],
    },
    {
        "query": "What are the exact inclusions for pre-hospitalization expenses?",
        "gold_sections": {"3.5"},
        "must_contain": ["60 days", "prior"],
    },
    {
        "query": "What are the exact inclusions for post-hospitalization expenses?",
        "gold_sections": {"3.6"},
        "must_contain": ["180 days", "discharge"],
    },
    {
        "query": "How to file reimbursement claims? Give me step wise complete procedure.",
        "gold_sections": {"11.6", "11.7"},
        "must_contain": ["30 days from the date of discharge", "claim form"],
    },
    {
        "query": "What is the procedure for cashless claims in India? Give me step wise complete procedure.",
        "gold_sections": {"11.2"},
        "must_contain": ["Network Provider", "pre-authorization"],
    },
    # Negative control — should route to no_answer:
    {
        "query": "What is the coverage for pet insurance under this policy?",
        "gold_sections": set(),
        "must_contain": [NO_ANSWER_TEXT],
    },
]

def evaluate(gold=GOLD):
    rows = []
    for case in gold:
        result = app.invoke({"query": case["query"]})
        got_secs = {c["section_id"] for c in result.get("reranked", [])}
        if case["gold_sections"]:
            recall = len(got_secs & case["gold_sections"]) / len(case["gold_sections"])
        else:
            recall = None
        ans = result["answer"].lower()
        hits = sum(kw.lower() in ans for kw in case["must_contain"])
        rows.append({
            "query": case["query"][:55] + "...",
            "section_recall": recall if recall is None else round(recall, 2),
            "keyword_hits": f"{hits}/{len(case['must_contain'])}",
            "routed": "no_answer" if is_no_answer(result["answer"]) else "answer",
            "top_score": round(max(result["scores"]), 2) if result.get("scores") else None,
            "context_tokens": sum(c["token_count"] for c in result.get("reranked", [])),
        })
    return pd.DataFrame(rows)

evaluate()


In [ ]:
# ================================================================
# 10.1) Threshold sweep — tune the gate against the gold set
# ================================================================
def sweep_thresholds(values=(-8, -6, -5, -4, -3, -2, -1, 0)):
    original = CONFIG["rerank_score_threshold"]
    rows = []
    for thr in values:
        CONFIG["rerank_score_threshold"] = thr
        df = evaluate()
        answered = (df["routed"] == "answer").sum()
        neg_ok = df.iloc[-1]["routed"] == "no_answer"   # negative control refused?
        rows.append({"threshold": thr, "answered": f"{answered}/{len(df)}",
                     "negative_control_refused": neg_ok})
    CONFIG["rerank_score_threshold"] = original
    return pd.DataFrame(rows)

# Pick the HIGHEST threshold that still answers all answerable queries
# while refusing the negative control.
# sweep_thresholds()


## 10.1 Regression check against the original notebook

`ORIGINAL_EVIDENCE` holds, for each of the original notebook's questions, the exact passages its
answers drew on (extracted by matching each original answer against the context the original
pipeline printed). Each passage is tagged with its section and whether that section is on-topic
for the question.

For every question this reports:
- **evidence_recall_all** — share of the original answer's evidence that is present in the v2.1
  context. The original pipeline scores 1.00 by construction. Target: ≥ 0.90.
- **evidence_recall_on_topic** — same, restricted to on-topic sections.
- **answer_grounding** — share of the new answer's wording found in its context (higher = less
  invented text). `orig_answer_grounding` is the original answer's score against its own context.

Read the last two columns together: several original answers were long but weakly grounded
(standard exclusions 9%, plan comparison 36%) — a shorter answer there is not a regression.

In [ ]:
# ================================================================
# 10.1) Regression check vs the original notebook's answers
# ================================================================
ORIGINAL_EVIDENCE = json.loads(r"""{
 "What is the difference between Optima Secure and Optima Super Secure plans? Give complete details.": {
  "passages": [
   {
    "section": "0",
    "on_topic": true,
    "text": "expenses incurred on hospitalization due to Illness or Accident."
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "offers additional coverage amount equivalent to 100%/200% of the Base Sum Insured."
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "offers additional coverage equivalent to 100% of the Base Sum Insured in 2 years irrespective of a claim."
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "restores 100% of Base Sum Insured automatically on partial or complete utilization of Sum Insured"
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "pays towards the Non-Medical expenses like gloves, food charges and other consumables during hospitalization."
   },
   {
    "section": "2.2",
    "on_topic": true,
    "text": "Maximum permissible amount for a single hospitalization claim:"
   },
   {
    "section": "2.2",
    "on_topic": true,
    "text": "Maximum permissible amount for all hospitalization claims in a policy year:"
   },
   {
    "section": "2.3",
    "on_topic": true,
    "text": "Maximum permissible amount for a single hospitalization claim:"
   },
   {
    "section": "2.3",
    "on_topic": true,
    "text": "Maximum permissible amount for all hospitalization claims in a policy year:"
   },
   {
    "section": "13.2",
    "on_topic": false,
    "text": "the base sum insured. The"
   }
  ],
  "orig_answer_grounding": 0.364
 },
 "What are the standard exclusions under the my:Optima Secure policy? Give complete details.": {
  "passages": [
   {
    "section": "10.1",
    "on_topic": true,
    "text": "All the Waiting Periods and exclusions listed below shall be applicable individually for each Insured Person and claims shall be assessed accordingly. a."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "Expenses related to the treatment of a pre-existing disease (PED) and its direct complications shall be excluded until the expiry of 36 months of continuous coverage after the date of inception of the first policy with"
   }
  ],
  "orig_answer_grounding": 0.094
 },
 "What are the exact standard exclusions under the my:Optima Secure policy? Give complete details.": {
  "passages": [
   {
    "section": "10.1",
    "on_topic": true,
    "text": "All the Waiting Periods and exclusions listed below shall be applicable individually for each Insured Person and claims shall be assessed accordingly. a."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "Expenses related to the treatment of a pre-existing disease (PED) and its direct complications shall be excluded until the expiry of 36 months of continuous coverage after the date of inception of the first policy with"
   }
  ],
  "orig_answer_grounding": 0.088
 },
 "What are the exact specific conditions with a 24-month waiting period? Give complete details.": {
  "passages": [
   {
    "section": "10.1",
    "on_topic": true,
    "text": "Expenses related to the treatment of"
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "shall be excluded until the expiry of"
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "months of continuous coverage after the date of inception of the first policy with"
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "If the Insured Person is continuously covered without any break as defined under the"
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "then waiting period for the same would be reduced to the extent of prior coverage."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "Expenses related to the treatment of the listed Conditions, surgeries/treatments shall be excluded until the expiry of 24 months of continuous coverage after the date of inception of the first Policy with"
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "This exclusion shall not be applicable for claims arising due to an Accident."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "any of the specified disease/procedure falls under the waiting period specified for Pre-Existing diseases, then the longer of the two waiting periods shall apply."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "The waiting period for listed conditions shall apply even if contracted after the Policy or declared and accepted without a specific exclusion."
   },
   {
    "section": "10.1",
    "on_topic": true,
    "text": "If the Insured Person is continuously covered without any break as defined under the applicable norms on portability stipulated by IRDAI, then waiting period for the same would be reduced to the extent of prior coverage."
   },
   {
    "section": "10.1.1",
    "on_topic": true,
    "text": "Joint Conditions: Non-infective Arthritis, Pilonidal Sinus, Osteoarthritis, Osteoporosis, Gout, Rheumatism. Gastrointestinal"
   },
   {
    "section": "10.1.1",
    "on_topic": true,
    "text": "Anorectal Conditions: Diseases of Gall Bladder including Cholecystitis, Pancreatitis, Ulcer and Erosion of Stomach and Duodenum, All Forms of Cirrhosis, Gastro Esophageal Reflux Disorder (GERD), Fissure"
   },
   {
    "section": "10.1.1",
    "on_topic": true,
    "text": "Fistula in Anus, Haemorrhoids, Perineal Abscesses, Perianal Abscesses."
   }
  ],
  "orig_answer_grounding": 0.469
 },
 "How to file reimbursement claims? Give me step wise complete procedure.": {
  "passages": [
   {
    "section": "11.3",
    "on_topic": false,
    "text": "as specified in the Policy,"
   },
   {
    "section": "11.5",
    "on_topic": false,
    "text": "days from the date of"
   },
   {
    "section": "11.6",
    "on_topic": true,
    "text": "submit the necessary documents to the Company within the prescribed time limit"
   },
   {
    "section": "11.6",
    "on_topic": true,
    "text": "Claim Types and Prescribed Time Limits: Reimbursement of Hospitalization, Day Care Treatment, or Pre-Hospitalization Expenses: Must be submitted within 30 days from the date of discharge from the hospital. Reimbursement of Post-Hospitalization Expenses: Must be submitted within 15 days from the completion of post-hospitalization treatment."
   },
   {
    "section": "11.7",
    "on_topic": true,
    "text": "The reimbursement claim is to be supported with the following documents"
   },
   {
    "section": "11.7",
    "on_topic": true,
    "text": "within the prescribed time limit. a. Duly Completed claim form, b. Photo ID and Age Proof, c. Copy of the Hospital's Registration Certificate/Hospital Registration number in case of Hospitalization in any non Network Provider of the Company or certificate from Hospital authorities providing facilities available including number of beds, d. Discharge Card"
   },
   {
    "section": "11.7",
    "on_topic": true,
    "text": "Transfer Summary, e. Final Hospital bill with all original deposit and final payment receipt and refund receipt(s), if advance amount refunded, f. Invoice with payment receipt and implant stickers for all implants used during Surgeries e.g. lens sticker and invoice in cataract Surgery, stent invoice and sticker in Angioplasty Surgery, g. All previous consultation papers indicating history and treatment details for current."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "for claims outside India of Insured Person and Accompanying Person (as applicable)"
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "Passport copy with entry and exit stamps"
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "Flight Tickets and Boarding Pass, if applicable"
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "Written advice from the overseas treating Medical Practitioner for requirement of an accompanying person during treatment."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "Any other relevant document required by Company for assessment of the claim."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "The Company shall only accept bills/invoices/medical treatment related documents only in the Insured Person's name for whom the claim is submitted."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "In the event of a claim lodged under the Policy and the original documents having been submitted to any other insurer, the Company shall accept the copy of the documents and claim settlement advice, duly certified by the other insurer subject to satisfaction of the Company."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "If requested by the Company, at the Company's cost, the Insured Person must submit to medical examination by Medical Practitioner appointed by the Company as often as it is considered reasonable and necessary and Company's representatives must be permitted to inspect the medical and Hospitalization records pertaining to the Insured Person's treatment, and to investigate the circumstances pertaining to the claim."
   },
   {
    "section": "11.7.1",
    "on_topic": true,
    "text": "Any delay in notification or submission may be condoned on merit where delay is proved to be for reasons beyond the control of the Insured Person. Section"
   },
   {
    "section": "12.2",
    "on_topic": false,
    "text": "within 30 days from the date of"
   }
  ],
  "orig_answer_grounding": 0.893
 },
 "What is the procedure for cashless claims in India? Give me step wise complete procedure.": {
  "passages": [
   {
    "section": "11.2",
    "on_topic": true,
    "text": "Procedure for Cashless Claims In India"
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "Cashless request form is available with the Network Provider."
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "The Network Provider shall obtain"
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "relevant information from the Insured"
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "and send a Cashless Facility request to the Company for authorization."
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "cashless request form and related medical information from the Insured"
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "pre-authorization letter to the Network Provider after verification."
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "At the time of discharge, the Insured Person shall verify and sign the discharge papers along with final bill, pay for non-medical and inadmissible expenses."
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "The Company reserves the right to deny pre-authorization in case the Insured Person is unable to provide the relevant medical details."
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "the Insured Person may obtain"
   },
   {
    "section": "11.2",
    "on_topic": true,
    "text": "treating doctor's advice and submit the claim documents to the Company for reimbursement."
   },
   {
    "section": "11.3",
    "on_topic": false,
    "text": "as specified in the Policy,"
   },
   {
    "section": "11.4",
    "on_topic": false,
    "text": "Procedure for Cashless Claims in"
   },
   {
    "section": "11.5",
    "on_topic": true,
    "text": "The Company reserves the right to"
   },
   {
    "section": "11.5",
    "on_topic": true,
    "text": "Pre-authorization issued by the Company shall be valid for 15 days from the date of issuance (or expiry of the Policy, whichever is earlier)."
   },
   {
    "section": "11.5",
    "on_topic": true,
    "text": "The Company shall make payment for the Cashless facility to the authorized amount, directly to the Network Provider."
   },
   {
    "section": "11.6",
    "on_topic": false,
    "text": "days from the date of"
   }
  ],
  "orig_answer_grounding": 0.743
 },
 "What is the Plus Benefit? Explain in detail.": {
  "passages": [
   {
    "section": "0",
    "on_topic": true,
    "text": "additional coverage amount equivalent to"
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "of the Base Sum Insured."
   },
   {
    "section": "0",
    "on_topic": true,
    "text": "equivalent to 100% of the Base Sum Insured in"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "the Policy is issued on an individual basis, the Plus Benefit shall be added and available individually to the Insured Person."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "floater basis, the Plus Benefit shall be added and available to all Family Members on a floater basis."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "if the Policy is renewed and due premium is received within the Grace Period."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "Renewed on a floater policy basis"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "the Plus Benefit to be carried forward for credit in"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "Renewed Policy shall be the lowest one that is applicable among all the Insured Persons."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "in cases where the Policy is split due to"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "child attaining the Age of 25 years, the Plus Benefit of the expiring policy shall be apportioned to such Renewed Policies in the proportion of the Sum Insured of each Renewed Policy"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "If the Sum Insured has been reduced at the time of Renewal, the applicable Plus Benefit shall be reduced in the same proportion to the Sum Insured in"
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "If the Sum Insured under the Policy has been increased at the time of Renewal, the Plus Benefit shall be calculated on the Sum Insured of the last completed Policy Year."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "If the Policy Period is of two or three years, the Plus Benefit shall be credited post completion of each Policy Year, and will be available for any claims made in the subsequent Policy Year."
   },
   {
    "section": "4.4.1",
    "on_topic": true,
    "text": "added to the Policy during subsequent Renewals will be eligible for the Plus Benefit as per their Renewal terms."
   }
  ],
  "orig_answer_grounding": 0.778
 },
 "What are the exact inclusions for pre-hospitalization expenses? Give complete details.": {
  "passages": [
   {
    "section": "3.1",
    "on_topic": false,
    "text": "Medicines and drugs prescribed in writing by"
   },
   {
    "section": "3.1",
    "on_topic": false,
    "text": "Intravenous fluids, blood transfusion, surgical appliances, allowable consumables and/or enteral feedings."
   },
   {
    "section": "3.1",
    "on_topic": false,
    "text": "The cost of prosthetics and other devices or equipment, if implanted internally during Surgery."
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "Expenses incurred on road Ambulance if the Insured Person is required to be transferred to the nearest Hospital for Emergency Care or from one Hospital to another Hospital or from Hospital to Home (within"
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "Care Dental Treatment, necessitated due to disease or Injury"
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "Plastic surgery, necessitated due to Injury"
   },
   {
    "section": "3.4",
    "on_topic": false,
    "text": "Medical Expenses incurred by the Insured Person"
   },
   {
    "section": "3.5",
    "on_topic": true,
    "text": "Pre-Hospitalization Medical Expenses incurred by the Insured Person, related to an admissible Hospitalization under Section 3.1 (Hospitalization Expenses), for up to 60 days immediately prior to the date of admissible Hospitalization covered under the Policy."
   },
   {
    "section": "3.6",
    "on_topic": false,
    "text": "Medical Expenses incurred by the Insured Person, related to an admissible Hospitalization under Section 3.1 (Hospitalization Expenses), for up to"
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "The organ donor's Pre-Hospitalization Expenses and Post Hospitalization Expenses."
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "Expenses related to organ transportation or preservation."
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "Any other Medical Expenses or Hospitalization consequent to the organ harvesting."
   },
   {
    "section": "4.2",
    "on_topic": false,
    "text": "if the Insured Person is"
   },
   {
    "section": "4.2",
    "on_topic": false,
    "text": "under Section 3.1 (Hospitalization Expenses)."
   },
   {
    "section": "10.2",
    "on_topic": false,
    "text": "laundry charges, attendant charges, ambulance collar, ambulance equipment, baby food, baby utility charges"
   }
  ],
  "orig_answer_grounding": 0.653
 },
 "What are the exact inclusions for post-hospitalization expenses? Give complete details.": {
  "passages": [
   {
    "section": "3.1",
    "on_topic": false,
    "text": "a. Room Rent, boarding, nursing expenses as provided by the Hospital"
   },
   {
    "section": "3.1",
    "on_topic": false,
    "text": "Nursing Home. b. Intensive Care Unit (ICU)"
   },
   {
    "section": "3.1",
    "on_topic": false,
    "text": "Intensive Cardiac Care Unit (ICCU) expenses. c. Surgeon, anaesthetist, Medical Practitioner, consultants, specialist Fees during Hospitalization forming part of Hospital bill. d. Investigative treatments and diagnostic procedures directly related to Hospitalization. e. Medicines and drugs prescribed in writing by Medical Practitioner. f. Intravenous fluids, blood transfusion, surgical appliances, allowable consumables and/or enteral feedings."
   },
   {
    "section": "3.1",
    "on_topic": false,
    "text": "The cost of prosthetics and other devices or equipment, if implanted internally during Surgery."
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "i. Expenses incurred on road Ambulance if the Insured Person is required to be transferred to the nearest Hospital for Emergency Care or from one Hospital to another Hospital or from Hospital to Home (within same city) following Hospitalization."
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "In patient Care Dental Treatment, necessitated due to disease or Injury"
   },
   {
    "section": "3.1.1",
    "on_topic": false,
    "text": "Plastic surgery, necessitated due to Injury"
   },
   {
    "section": "3.4",
    "on_topic": false,
    "text": "Medical Expenses incurred by the Insured Person"
   },
   {
    "section": "3.5",
    "on_topic": false,
    "text": "Medical Expenses incurred by the Insured Person, related to an admissible Hospitalization under Section 3.1 (Hospitalization Expenses), for up to"
   },
   {
    "section": "3.6",
    "on_topic": true,
    "text": "Medical Expenses incurred by the Insured Person, related to an admissible Hospitalization under Section 3.1 (Hospitalization Expenses), for up to 180 days from the date of discharge from the Hospital,"
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "incurred by the Insured Person"
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "under Section 3.1 (Hospitalization Expenses)"
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "The organ donor's Pre-Hospitalization Expenses and Post Hospitalization Expenses."
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "Expenses related to organ transportation or preservation."
   },
   {
    "section": "3.7",
    "on_topic": false,
    "text": "Any other Medical Expenses or Hospitalization consequent to the organ harvesting."
   },
   {
    "section": "4.2",
    "on_topic": false,
    "text": "if the Insured Person is"
   },
   {
    "section": "4.2",
    "on_topic": false,
    "text": "under Section 3.1 (Hospitalization Expenses)."
   },
   {
    "section": "4.3",
    "on_topic": false,
    "text": "Non-Medical Expenses listed under Annexure B to this Policy incurred in relation to a claim admissible under Section 3 (Base Coverage) during the Policy Year. Exclusion (k) of Section 10.2"
   },
   {
    "section": "4.3",
    "on_topic": false,
    "text": "Specific Exclusions shall not apply to this Cover."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "The provision or fitting of hearing aids, spectacles or contact lenses."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "expenses for alopecia, baldness including corticosteroids and topical immunotherapy wigs, toupees, hair pieces, any non-surgical hair replacement methods, optometric therapy."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Artificial limbs and/or device used for diagnosis or treatment (except when used intra-operatively), prosthesis, corrective devices external durable medical equipment of any kind, wheelchairs, crutches, and oxygen concentrator for bronchial asthma/ COPD conditions, cost of cochlear implant(s) unless necessitated by an Accident."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Any treatment or part of a treatment that is not of a reasonable charge and not Medically Necessary. Drugs or treatments which are not supported by a prescription."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Any permanent exclusion applied on any medical or physical condition or treatment of an Insured Person as specifically mentioned in the Policy Schedule and as specifically accepted by Policyholder/Insured Person."
   }
  ],
  "orig_answer_grounding": 0.808
 },
 "What expenses are covered under the Overseas Travel Secure option? Give complete details.": {
  "passages": [
   {
    "section": "4.7",
    "on_topic": false,
    "text": "Section 4.9 (Global Health Cover (Emergency Treatments Only)),"
   },
   {
    "section": "4.10.1",
    "on_topic": false,
    "text": "Only those Pre-hospitalization Medical Expenses and/or Post-hospitalization Medical Expenses shall be admissible under this benefit that have been incurred and paid overseas. Such expenses should be related to an admissible overseas Hospitalization claim only (as per details in invoice/supporting documents)."
   },
   {
    "section": "4.11",
    "on_topic": true,
    "text": "conditions are fulfilled: a. The overseas treating Medical Practitioner has advised a minimum hospitalization of 5 consecutive days and has also advised the requirement of an accompanying person during treatment. b. We have accepted a claim under"
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Treatment taken on outpatient basis."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "The provision or fitting of hearing aids, spectacles or contact lenses."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Any treatment and associated expenses for alopecia, baldness"
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "corticosteroids and topical immunotherapy wigs, toupees, hair pieces, any non-surgical hair replacement methods, optometric therapy."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Expenses for Artificial limbs and/or device used for diagnosis or treatment (except when used intra-operatively), prosthesis, corrective devices external durable medical equipment of any kind, wheelchairs, crutches, and oxygen concentrator for bronchial asthma/ COPD conditions, cost of cochlear implant(s) unless necessitated by an Accident."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Any treatment or part of a treatment that is not of a reasonable charge and not Medically Necessary. Drugs or treatments which are not supported by a prescription."
   },
   {
    "section": "10.2.1",
    "on_topic": false,
    "text": "Any permanent exclusion applied on any medical or physical condition or treatment of an Insured Person as specifically mentioned in the Policy Schedule and as specifically accepted by Policyholder/Insured Person. Such exclusions"
   }
  ],
  "orig_answer_grounding": 0.832
 }
}""")

def _key(w: str) -> str:
    return re.sub(r"[^\w₹%]", "", w.lower())

def _keys(text: str):
    return [k for k in (_key(w) for w in text.split()) if k]

def _grams(keys, n=5):
    return {tuple(keys[i:i + n]) for i in range(len(keys) - n + 1)}

def _covered(keys, grams, n=5):
    cov = [False] * len(keys)
    for i in range(len(keys) - n + 1):
        if tuple(keys[i:i + n]) in grams:
            for j in range(i, i + n):
                cov[j] = True
    return sum(cov), len(keys)

def evidence_recall(text: str, passages, n=5):
    g = _grams(_keys(text), n)
    hit = tot = 0
    for p in passages:
        h, t = _covered(_keys(p["text"]), g, n)
        hit, tot = hit + h, tot + t
    return round(hit / tot, 2) if tot else None

def grounding(answer: str, context: str, n=5):
    h, t = _covered(_keys(answer), _grams(_keys(context), n), n)
    return round(h / t, 2) if t else None

def regression_check():
    rows = []
    for q, fx in ORIGINAL_EVIDENCE.items():
        res = app.invoke({"query": q})
        ctx = "\n\n".join(c["text"] for c in res.get("reranked", []))
        on = [p for p in fx["passages"] if p["on_topic"]]
        rows.append({
            "query": q[:50] + "...",
            "context_tokens": sum(c["token_count"] for c in res.get("reranked", [])),
            "evidence_recall_all": evidence_recall(ctx, fx["passages"]),
            "evidence_recall_on_topic": evidence_recall(ctx, on) if on else None,
            "answer_grounding": grounding(res["answer"], ctx),
            "orig_answer_grounding": fx["orig_answer_grounding"],
        })
    return pd.DataFrame(rows)

regression_check()


## 10.2 Generation A/B on identical contexts (optional, ~3× the LLM calls)

Retrieval is run once per question; then the same context is answered three ways:
- **A_original** — original prompt, raw (no chat template): the original notebook's generation
- **B_v2.1** — original prompt + section-labeled blocks + chat template (v2.1 default)
- **C_v2_prompt** — v2's modified instructions, for reference

`answer_evidence_recall` = how much of the original answer's evidence each new answer reproduces
(completeness relative to the original); `grounding` = share of the answer found in the context
(faithfulness). Pick the variant with the best completeness at equal or better grounding, then set
`CONFIG["use_chat_template"]` accordingly.

In [ ]:
# ================================================================
# 10.2) Generation A/B — same retrieved context, three prompt variants
# ================================================================
V2_PROMPT_TEMPLATE = PROMPT_TEMPLATE.replace(
    "Use ONLY the information provided in the CONTEXT to answer the QUESTION.\n",
    "Use ONLY the information provided in the CONTEXT to answer the QUESTION.\n"
    "Only use context blocks that are actually relevant to the question; ignore unrelated sections.\n"
).replace(
    "When relevant, cite the section name or number from the CONTEXT.",
    "Cite the section number shown in brackets for each fact you state.")

def generation_ab(questions=None):
    rows = []
    for q in (questions or list(ORIGINAL_EVIDENCE)):
        st = retrieve_node({"query": q}); st["query"] = q
        st.update(rerank_node(st))
        chunks = st["reranked"]
        ctx_plain = "\n\n".join(c["text"] for c in chunks)
        passages = ORIGINAL_EVIDENCE.get(q.strip(), {}).get("passages", [])
        variants = {
            "A_original": (PROMPT_TEMPLATE.format(context=ctx_plain, query=q.strip()), False),
            "B_v2.1": (build_prompt(q, chunks), True),
            "C_v2_prompt": (V2_PROMPT_TEMPLATE.format(context=format_context(chunks), query=q.strip()), True),
        }
        for name, (prompt, tpl) in variants.items():
            ans = generate_answer(prompt, use_chat_template=tpl)
            rows.append({"query": q[:40] + "...", "variant": name,
                         "answer_words": len(ans.split()),
                         "answer_evidence_recall": evidence_recall(ans, passages) if passages else None,
                         "grounding": grounding(ans, ctx_plain)})
    return pd.DataFrame(rows)

# ab = generation_ab(); ab.pivot(index="query", columns="variant",
#                                values=["answer_evidence_recall", "grounding"])


## 11. Notes for going further

- **Same fixes apply to the v3 (agentic) notebook**, which reuses v2's parsing, chunking and
  context code unchanged underneath.
- **If CUDA out-of-memory persists** after restarting the runtime: check the `[GPU …]` line
  printed after model loading. If an "LLM modules were placed on CPU/disk" warning appears, or
  free memory is under ~1 GB, set `CONFIG["llm_load_in_4bit"] = True` and rerun from the top.
  4-bit changes the model's arithmetic slightly, so rerun Section 10.1 afterwards to confirm the
  answers still match the original's evidence.
- **Query rewriting:** if eval shows recall misses on paraphrased user language, add one
  LangGraph retry loop that rewrites the query with the LLM when the gate fails.
- **Multi-policy scale:** move to a vector store with metadata filtering (Qdrant / pgvector)
  so each query searches only its own policy — a correctness requirement, not performance.
- **Faithfulness scoring:** add LLM-as-judge (e.g., RAGAS) once the gold set stabilizes.
- **Serving split:** Sections 1–4 are ingestion; 5–9 are serving. The persisted FAISS index and
  chunks.json are the handoff artifacts when moving off Colab.
